# Caregiver AI Lab — Complete Colab Notebook

Everything in one place: synthetic dataset, baseline, generative model, evaluation, error analysis, report, and Gradio UI. **No separate `.py` files needed.**

⚠️ **Not a medical tool.** Synthetic fictional data only. Never diagnose or recommend treatment.

In [1]:

import os, sys, json, re, random, time, statistics, csv, hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

ROOT = Path("/content/caregiver-ai-lab")
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
DATASET_PATH = DATA_DIR / "synthetic_examples.jsonl"
RESULTS_PATH = RESULTS_DIR / "experiment_results.jsonl"
REPORT_PATH = RESULTS_DIR / "experiment_report.md"
REVIEW_PATH = RESULTS_DIR / "human_review_export.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Project root:", ROOT)
print("HF_TOKEN configured:", bool(os.getenv("HF_TOKEN")))

Python: 3.13.15
Project root: /content/caregiver-ai-lab
HF_TOKEN configured: False


In [2]:

DEFAULT_MODEL_ID = os.getenv("CAREGIVER_MODEL_ID", "Omartificial-Intelligence-Project/AraT5v2-base-102M")
BASELINE_MODEL_ID = "rule-based-extractive-baseline"
MAX_NEW_TOKENS = int(os.getenv("CAREGIVER_MAX_NEW_TOKENS", "256"))
MAX_INPUT_CHARS = int(os.getenv("CAREGIVER_MAX_INPUT_CHARS", "2000"))
SYNTHETIC_DATASET_VERSION = "v1.0-synthetic"
SEED = 42
print("Default model:", DEFAULT_MODEL_ID)

Default model: Omartificial-Intelligence-Project/AraT5v2-base-102M


In [3]:

LANGUAGE_TYPES = ("ar", "mix", "en")
REQUIRED_FIELDS = ["example_id","language_type","input_note","expected_facts",
                   "important_details","uncertainties","known_contradictions",
                   "expected_summary","expected_evidence_quotes"]

CLAUSES: Dict[str, List[Tuple[str,str,str,str,bool]]] = {
    "appetite": [
        ("أكلت وجبة الغداء جيداً","ate lunch well","ate lunch well","appetite",False),
        ("أكلت كمية قليلة جداً","ate a very small amount","ate small amount","appetite",False),
        ("لم تأكل شيئًا منذ الصباح","has not eaten anything since morning","not eaten since morning","appetite",False),
        ("غير متأكدين إذا كانت أكلت أم لا","unsure whether she ate or not","uncertain whether ate","appetite",True),
    ],
    "medication": [
        ("أخدت الدواء في الموعد","took the medication on time","medication taken on time","medication",False),
        ("لم تتناول الدواء بعد","has not taken the medication yet","medication not taken","medication",False),
        ("مش متأكدين إذا أخذت الدواء ولا لأ","unsure if she took the medication or not","uncertain medication intake","medication",True),
        ("الدواء موجود في الحقيبة","the medication is in the bag","medication present in bag","medication",False),
    ],
    "sleep": [
        ("نمت جيداً طوال الليل","slept well all night","slept well all night","sleep",False),
        ("النوم كان متقطعاً","sleep was interrupted","sleep interrupted","sleep",False),
        ("لم تنم جيداً أمس","did not sleep well yesterday","did not sleep well","sleep",False),
        ("غير متأكدين إذا نمت أو سهرت","unsure whether she slept or stayed awake","uncertain sleep","sleep",True),
    ],
    "mobility": [
        ("تمشي بمفردها داخل البيت","walks alone inside the house","walks alone inside house","mobility",False),
        ("تحتاج مساعدة للمشي","needs assistance to walk","needs assistance walking","mobility",False),
        ("تستخدم كرسي متحرك","uses a wheelchair","uses wheelchair","mobility",False),
        ("مش متأكدين إذا تقدر تمشي أم لأ","unsure whether she can walk or not","uncertain mobility","mobility",True),
    ],
    "pain": [
        ("تشكو من ألم في الركبة","complains of knee pain","knee pain","pain",False),
        ("لا تشكو من أي ألم","does not complain of any pain","no pain","pain",False),
        ("ألم خفيف في البطن يأتي ويذهب","mild stomach pain that comes and goes","mild intermittent stomach pain","pain",False),
        ("غير متأكدين إذا كانت تعاني من ألم","unsure whether she is in pain","uncertain pain","pain",True),
    ],
    "mood": [
        ("مزاجها جيد ومتفائلة","mood is good and she is optimistic","good mood","mood",False),
        ("تبدو حزينة ومقفلة","appears sad and withdrawn","sad and withdrawn","mood",False),
        ("متقلبة المزاج اليوم","mood is changeable today","changeable mood","mood",False),
    ],
    "toileting": [
        ("استخدمت الحمام بشكل طبيعي","used the bathroom normally","used bathroom normally","toileting",False),
        ("تحتاج مساعدة لدخول الحمام","needs assistance to use the bathroom","needs bathroom assistance","toileting",False),
        ("لم تذهب للحمام منذ الصباح","has not gone to the bathroom since morning","no bathroom visit since morning","toileting",False),
    ],
    "hydration": [
        ("شربت ماءً كافٍ اليوم","drank enough water today","drank enough water","hydration",False),
        ("شربت كمية قليلة من الماء","drank a small amount of water","drank little water","hydration",False),
        ("غير متأكدين إذا شربت الماء أم لا","unsure whether she drank water or not","uncertain water intake","hydration",True),
    ],
}
IRRELEVANT_CLAUSES = [
    ("الطقس معتدل اليوم","the weather is mild today","weather mild","irrelevant",False),
    ("البيت نظيف ومرتب","the house is clean and tidy","house clean","irrelevant",False),
    ("شاهدت التلفزيون مساء أمس","watched television last night","watched tv","irrelevant",False),
]
NAMES_AR = ["أم خالد","سارة","جدّي","ماما","الأخت منى","عم يوسف","التيتو","خالتي","خالد","نورا"]
NAMES_EN = ["Um Khalid","Sara","Grandpa","Mama","Aunt Mona","Uncle Youssef","Teta","Khalti","Khalid","Nora"]
SUMMARY_TEMPLATES_AR = ["{name}: {facts}.","ملخص حالة {name}: {facts}.","{name} case summary: {facts}."]
SUMMARY_TEMPLATES_EN = ["{name}: {facts}.","Care summary for {name}: {facts}.","{name} note summary: {facts}."]
INTRO_AR = "ملاحظة رعاية يومية خيالية لـ {name}."
INTRO_EN = "Fictional daily care note for {name}."

def _rng_for(example_id: str) -> random.Random:
    return random.Random(hash(example_id) & 0xFFFFFFFF)

def _pick(pool: List[Tuple], rng: random.Random, k: int) -> List[Tuple]:
    return rng.sample(pool, min(k, len(pool)))

def _contradiction_pair(category: str) -> Optional[Tuple[Tuple, Tuple]]:
    opposites = {
        "appetite": ("أكلت جيدًا","لم تأكل شيئًا"),
        "sleep": ("نمت جيدًا","لم تنم جيدًا"),
        "mobility": ("تمشي بمفردها","تحتاج مساعدة للمشي"),
        "pain": ("لا تشكو من أي ألم","تشكو من ألم"),
        "hydration": ("شربت ماءً كافٍ","لم تشرب ماءً"),
    }
    if category not in opposites:
        return None
    a_text, b_text = opposites[category]
    ar_a = next(c[0] for c in CLAUSES[category] if c[0].startswith(a_text.split(" ")[0]))
    ar_b = next(c[0] for c in CLAUSES[category] if not c[0].startswith(a_text.split(" ")[0]))
    return (ar_a, ar_b)

def generate_example(idx: int) -> Dict[str, Any]:
    rng = _rng_for(f"EG-SYN-{idx:04d}")
    lang = LANGUAGE_TYPES[idx % 3]
    name_ar = rng.choice(NAMES_AR)
    name_en = rng.choice(NAMES_EN)
    categories = rng.sample(list(CLAUSES.keys()), k=rng.randint(2, 4))
    chosen: List[Tuple] = [rng.choice(CLAUSES[cat]) for cat in categories]
    contradictions: List[Tuple[str,str]] = []
    if rng.random() < 0.25:
        pair = _contradiction_pair(rng.choice(list(CLAUSES.keys())))
        if pair:
            chosen.extend(list(pair))
            contradictions.append((pair[0][0], pair[1][0]))
    uncertain_clauses: List[str] = []
    if rng.random() < 0.35:
        uncertain_clauses = [c[0] for c in chosen[:] if c[4]]
    irrelevant: List[Tuple] = []
    if rng.random() < 0.4:
        irrelevant = _pick(IRRELEVANT_CLAUSES, rng, rng.randint(1, 2))
    care_ar = [c[0] for c in chosen]
    care_en = [c[1] for c in chosen]
    facts = [c[2] for c in chosen]
    note_ar = f"{INTRO_AR.format(name=name_ar)} " + " ".join(care_ar)
    note_en = f"{INTRO_EN.format(name=name_en)} " + " ".join(care_en)
    if irrelevant:
        irr = irrelevant[0]
        note_ar += " " + irr[0]
        note_en += " " + irr[1]
    if lang == "ar":
        input_note = note_ar
    elif lang == "en":
        input_note = note_en
    else:
        half = max(1, len(care_ar) // 2)
        input_note = " ".join(care_ar[:half]) + " " + " ".join(care_en[half:]) + "."
    important_details = [f for f in facts if not f.startswith(("weather","house","watched"))]
    expected_evidence_quotes = [c[0] for c in chosen]
    if irrelevant:
        expected_evidence_quotes.extend([c[0] for c in irrelevant])
    fact_list_ar = "، ".join(care_ar) if care_ar else "لا توجد ملاحظات واضحة"
    fact_list_en = "; ".join(care_en) if care_en else "no clear observations"
    if lang == "ar":
        expected_summary = rng.choice(SUMMARY_TEMPLATES_AR).format(name=name_ar, facts=fact_list_ar)
    elif lang == "en":
        expected_summary = rng.choice(SUMMARY_TEMPLATES_EN).format(name=name_en, facts=fact_list_en)
    else:
        expected_summary = f"{name_en} / {name_ar}: {fact_list_en}."
    has_medication = any(c[3] == "medication" for c in chosen)
    return {
        "example_id": f"EG-SYN-{idx:04d}",
        "language_type": lang,
        "input_note": input_note,
        "expected_facts": facts,
        "important_details": important_details,
        "uncertainties": uncertain_clauses,
        "known_contradictions": contradictions,
        "expected_summary": expected_summary,
        "expected_evidence_quotes": expected_evidence_quotes,
        "_synthetic": True,
        "_dataset_version": SYNTHETIC_DATASET_VERSION,
        "_has_medication_info": has_medication,
    }

def generate_dataset(n: int = 150) -> List[Dict[str, Any]]:
    return [generate_example(i) for i in range(1, n + 1)]

def write_dataset(examples: List[Dict[str, Any]], path: Path = DATASET_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for ex in examples:
            fh.write(json.dumps(ex, ensure_ascii=False) + "\n")

def load_dataset(path: Path = DATASET_PATH) -> List[Dict[str, Any]]:
    examples: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                examples.append(json.loads(line))
    return examples

def validate_dataset(examples: List[Dict[str, Any]]) -> Dict[str, Any]:
    errors: List[str] = []
    seen_ids: set = set()
    malformed = 0
    for idx, ex in enumerate(examples):
        label = ex.get("example_id", f"row-{idx}")
        if not isinstance(ex, dict):
            malformed += 1
            errors.append(f"{label}: entry is not a JSON object")
            continue
        for field in REQUIRED_FIELDS:
            value = ex.get(field)
            if value is None or value == "":
                errors.append(f"{label}: missing or empty field '{field}'")
            if field == "expected_facts" and not isinstance(value, list):
                errors.append(f"{label}: 'expected_facts' must be a list")
            if field == "language_type" and value not in LANGUAGE_TYPES:
                errors.append(f"{label}: invalid language_type '{value}'")
        eid = ex.get("example_id")
        if eid:
            if eid in seen_ids:
                errors.append(f"duplicate example_id: {eid}")
            seen_ids.add(eid)
        if not ex.get("input_note"):
            malformed += 1
    return {
        "valid": len(errors) == 0 and malformed == 0,
        "count": len(examples),
        "unique_ids": len(seen_ids),
        "duplicate_ids": len(seen_ids) != len(examples),
        "malformed": malformed,
        "errors": errors[:50],
    }
print("Dataset generator defined.")

Dataset generator defined.


In [4]:

_HARAKAT = re.compile(r"[\u064B-\u065F]")
_TATWEEL = re.compile(r"ـ+")
_MULTISPACE = re.compile(r"\s+")
_ARABIC_PUNCT_MAP = {"،": ",", "؛": ";", "؟": "?", "٫": ".", "：": ":"}
_ALEF_MAP = str.maketrans({"إ": "ا", "أ": "ا", "آ": "ا"})
_YE_MAP = str.maketrans({"ى": "ي", "ؤ": "و", "ئ": "ي"})
_TEH_MARBUTA_MAP = str.maketrans({"ة": "ه"})

def normalize_arabic(text: str) -> str:
    if not text:
        return text
    text = _HARAKAT.sub("", text)
    text = _TATWEEL.sub("", text)
    text = text.translate(_ALEF_MAP)
    text = text.translate(_YE_MAP)
    text = text.translate(_TEH_MARBUTA_MAP)
    for ar, en in _ARABIC_PUNCT_MAP.items():
        text = text.replace(ar, en)
    text = _MULTISPACE.sub(" ", text).strip()
    return text

def normalize(text: str, arabic: bool = True) -> str:
    if not text:
        return text
    if arabic:
        text = normalize_arabic(text)
    return text.lower()

def split_sentences(text: str) -> List[str]:
    if not text:
        return []
    parts = re.split(r"(?<=[.!?،؛؟])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def word_tokens(text: str, arabic: bool = True) -> List[str]:
    return normalize(text, arabic=arabic).split()

def content_words(text: str, arabic: bool = True) -> List[str]:
    return [w for w in word_tokens(text, arabic=arabic) if len(w) > 2]
print("Normalization defined.")

Normalization defined.


In [5]:

from dataclasses import dataclass, field

class Observation(BaseModel):
    category: str = Field(..., min_length=1)
    statement: str = Field(..., min_length=1)
    evidence_quote: str = Field(default="")
    uncertain: bool = Field(default=False)

class UnclearOrConflicting(BaseModel):
    statement: str = Field(..., min_length=1)
    evidence_quote: str = Field(default="")

class UnsupportedClaim(BaseModel):
    statement: str = Field(..., min_length=1)
    reason: str = Field(default="no supporting evidence in the input note")

class CareSummary(BaseModel):
    summary: str = Field(..., min_length=1)
    observations: List[Observation] = Field(default_factory=list)
    unclear_or_conflicting: List[UnclearOrConflicting] = Field(default_factory=list)
    unsupported_claims: List[UnsupportedClaim] = Field(default_factory=list)
    model_id: Optional[str] = None
    latency_s: Optional[float] = None
    valid_json: bool = True
    parse_error: Optional[str] = None

    def to_flat_text(self) -> str:
        parts = [self.summary]
        for o in self.observations:
            parts.extend([o.statement, o.evidence_quote])
        for c in self.unclear_or_conflicting:
            parts.extend([c.statement, c.evidence_quote])
        for u in self.unsupported_claims:
            parts.append(u.statement)
        return " ".join(p for p in parts if p)


@dataclass
class ModelResult:
    model_id: str
    input_note: str
    generation_config: Dict[str, Any]
    summary: str
    structured: Dict[str, Any] = field(default_factory=dict)
    evidence_quotes: List[str] = field(default_factory=list)
    latency_s: float = 0.0
    error: Optional[str] = None
    ok: bool = True
    valid_json: bool = True
    def to_record(self) -> Dict[str, Any]:
        return {"model_id": self.model_id, "input_note": self.input_note,
                "generation_config": self.generation_config, "summary": self.summary,
                "structured": self.structured, "evidence_quotes": self.evidence_quotes,
                "latency_s": self.latency_s, "error": self.error, "ok": self.ok,
                "valid_json": self.valid_json}
print("Schemas defined.")

Schemas defined.


In [6]:

CATEGORY_KEYWORDS: Dict[str, List[str]] = {
    "appetite": ["أكل","طعام","وجبة","شهية","ate","eat","food","lunch","breakfast","dinner"],
    "medication": ["دواء","medication","medicine","drug","pill"],
    "sleep": ["نوم","سهر","نمت","sleep","slept","awake"],
    "mobility": ["تمشي","مشي","كرسي","مساعدة","walk","walks","wheelchair","mobility"],
    "pain": ["ألم","وجع","مؤلم","pain","hurt","hurts"],
    "mood": ["مزاج","حزينة","متفائلة","mood","sad","optimistic","withdrawn"],
    "toileting": ["حمام","الحمام","دخول الحمام","bathroom","toilet"],
    "hydration": ["ماء","شربت","ماء كاف","water","drank","thirsty"],
}
UNCERTAINTY_MARKERS = ["مش متأكد","غير متأكد","لا ندري","غير واضحة","لا نعلم","ربما","ممكن",
    "unsure","uncertain","unknown","maybe","possibly","not sure","unclear"]
OPPOSITE_STARTS = [("أكلت","لم تأكل"),("نمت","لم تنم"),("تمشي","تحتاج مساعدة"),
                   ("لا تشكو","تشكو"),("شربت","لم تشرب")]

def _category_score(sentence: str) -> Tuple[str, int]:
    norm = normalize(sentence)
    best_cat, best_score = "general", 0
    for cat, keywords in CATEGORY_KEYWORDS.items():
        score = sum(1 for kw in keywords if normalize(kw) in norm)
        if score > best_score:
            best_cat, best_score = cat, score
    return best_cat, best_score

def _has_uncertainty(sentence: str) -> bool:
    norm = normalize(sentence)
    return any(normalize(m) in norm for m in UNCERTAINTY_MARKERS)

def _detect_contradictions(note: str) -> List[str]:
    norm = normalize(note)
    flags: List[str] = []
    for a, b in OPPOSITE_STARTS:
        if normalize(a) in norm and normalize(b) in norm:
            flags.append(f"contradiction between '{a}' and '{b}'")
    return flags

class RuleBasedBaseline:
    def __init__(self, model_id: str = BASELINE_MODEL_ID):
        self.model_id = model_id
    def name(self) -> str:
        return self.model_id
    def available(self) -> bool:
        return True
    def predict(self, note: str, **kwargs) -> ModelResult:
        start = time.time()
        config = {"approach": "extractive-rule-based", "top_k": 6}
        try:
            sentences = [s for s in split_sentences(note) if s.strip()]
            if not sentences:
                structured = CareSummary(summary="Empty note.", observations=[], unclear_or_conflicting=[], unsupported_claims=[], model_id=self.model_id, valid_json=True).model_dump()
                latency = round(time.time() - start, 4)
                return ModelResult(model_id=self.model_id, input_note=note, generation_config=config, summary="Empty note.", structured=structured, evidence_quotes=[], latency_s=latency, ok=True, valid_json=True)
            scored = []
            for s in sentences:
                cat, score = _category_score(s)
                scored.append((score, s, cat))
            scored.sort(key=lambda x: x[0], reverse=True)
            best_score = scored[0][0]
            candidates = scored if best_score <= 0 else [item for item in scored if item[0] > 0]
            top = candidates[: kwargs.get("top_k", 6)]
            observations = []
            for score, sentence, cat in top:
                observations.append(Observation(category=cat, statement=sentence, evidence_quote=sentence, uncertain=_has_uncertainty(sentence)).model_dump())
            contradictions = _detect_contradictions(note)
            unclear = []
            norm_sentences = [normalize(s) for s in sentences]
            for c in contradictions:
                keys = c.replace("contradiction between '", "").split("' and '")
                for s, ns in zip(sentences, norm_sentences):
                    if any(normalize(k) in ns for k in keys):
                        unclear.append(UnclearOrConflicting(statement=c, evidence_quote=s).model_dump())
                        break
            summary_parts = [o["statement"] for o in observations] or [note.strip() or "No content provided."]
            summary = " ".join(summary_parts)
            structured = CareSummary(summary=summary, observations=observations, unclear_or_conflicting=unclear, unsupported_claims=[], model_id=self.model_id, valid_json=True).model_dump()
            latency = round(time.time() - start, 4)
            return ModelResult(model_id=self.model_id, input_note=note, generation_config=config, summary=summary, structured=structured, evidence_quotes=[o.get("evidence_quote","") for o in observations], latency_s=latency, ok=True, valid_json=True)
        except Exception as exc:
            latency = round(time.time() - start, 4)
            return ModelResult(model_id=self.model_id, input_note=note, generation_config=config, summary="", structured={}, evidence_quotes=[], latency_s=latency, ok=False, error=str(exc), valid_json=True)
print("Baseline defined.")

Baseline defined.


In [7]:

_JSON_BLOCK = re.compile(r"\{.*\}", re.DOTALL)

def _extract_json_block(text: str) -> Optional[str]:
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    return None

def _parse_structured(raw: str) -> Tuple[Dict[str, Any], bool, Optional[str]]:
    block = _extract_json_block(raw)
    if block is None:
        return {"raw_output": raw.strip()}, False, "no JSON object found in model output"
    try:
        data = json.loads(block)
        validated = CareSummary(**data)
        return validated.model_dump(), True, None
    except Exception as exc:
        return {"raw_output": raw.strip(), "parse_detail": block[:200]}, False, str(exc)

class HfGenerativeModel:
    def __init__(self, model_id: Optional[str] = None):
        self.model_id = model_id or DEFAULT_MODEL_ID
        self._pipe: Any = None
        self._load_error: Optional[str] = None
        self._tried_load = False
    def name(self) -> str:
        return self.model_id
    def available(self) -> bool:
        if self._pipe is not None:
            return True
        if self._tried_load and self._load_error:
            return False
        try:
            from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
            self._tried_load = True
            tokenizer = AutoTokenizer.from_pretrained(self.model_id, token=os.getenv("HF_TOKEN") or None)
            model = AutoModelForSeq2SeqLM.from_pretrained(self.model_id, token=os.getenv("HF_TOKEN") or None)
            device = 0 if __import__("torch").cuda.is_available() else -1
            self._pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=device)
            return True
        except Exception as exc:
            self._tried_load = True
            self._load_error = str(exc)
            return False
    def predict(self, note: str, **kwargs) -> ModelResult:
        start = time.time()
        config = {"model_id": self.model_id, "max_new_tokens": kwargs.get("max_new_tokens", MAX_NEW_TOKENS), "truncation": True}
        try:
            if not self.available():
                raise RuntimeError(self._load_error or "model failed to load")
            text = note[:MAX_INPUT_CHARS]
            out = self._pipe(text, max_new_tokens=config["max_new_tokens"], truncation=True)
            generated = ""
            if isinstance(out, list) and out:
                first = out[0]
                generated = first.get("summary_text") or first.get("generated_text") or str(first)
            generated = str(generated).strip()
            structured, valid_json, parse_error = _parse_structured(generated)
            summary = structured.get("summary", generated) if isinstance(structured, dict) else generated
            evidence = [o.get("evidence_quote","") for o in structured.get("observations", [])] if isinstance(structured, dict) else []
            latency = round(time.time() - start, 4)
            return ModelResult(model_id=self.model_id, input_note=note, generation_config=config, summary=summary, structured=structured, evidence_quotes=[e for e in evidence if e], latency_s=latency, ok=True, valid_json=valid_json, error=parse_error)
        except Exception as exc:
            latency = round(time.time() - start, 4)
            return ModelResult(model_id=self.model_id, input_note=note, generation_config=config, summary="", structured={}, evidence_quotes=[], latency_s=latency, ok=False, error=str(exc), valid_json=False)
print("HF generative model defined.")

HF generative model defined.


In [8]:

def quote_found_in_note(quote: str, note: str) -> bool:
    if not quote or not note:
        return False
    q = normalize(quote)
    n = normalize(note)
    if not q:
        return False
    if q in n:
        return True
    words = word_tokens(quote)
    if not words:
        return False
    return all(w in n for w in words)

UNCERTAINTY_MARKERS_AR = ["مش متأكد","غير متأكد","لا ندري","غير واضحة","لا نعلم","ربما","ممكن","غالباً","أعتقد","لا أعرف"]
UNCERTAINTY_MARKERS_EN = ["unsure","uncertain","unknown","maybe","possibly","probably","not sure","unclear","i think"]
_ALL_MARKERS = [normalize(m) for m in UNCERTAINTY_MARKERS_AR + UNCERTAINTY_MARKERS_EN]

def has_uncertainty_marker(text: str) -> bool:
    if not text:
        return False
    norm = normalize(text)
    return any(m in norm for m in _ALL_MARKERS)
print("Evidence + uncertainty helpers defined.")

Evidence + uncertainty helpers defined.

In [9]:

def _as_list(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    return [str(v) for v in value]

def _output_text(result: Dict[str, Any]) -> str:
    structured = result.get("structured", {}) or {}
    parts = [structured.get("summary", ""), result.get("summary", "")]
    for obs in structured.get("observations", []) or []:
        parts.extend([obs.get("statement", ""), obs.get("evidence_quote", "")])
    for c in structured.get("unclear_or_conflicting", []) or []:
        parts.extend([c.get("statement", ""), c.get("evidence_quote", "")])
    return " ".join(p for p in parts if p)

def _fact_recall_example(expected_facts: List[str], output_text: str) -> Optional[float]:
    if not expected_facts:
        return None
    norm_output = output_text.lower()
    hits = 0
    for fact in expected_facts:
        words = word_tokens(fact)
        if not words:
            continue
        if all(w in norm_output for w in words):
            hits += 1
    return hits / len(expected_facts)

def json_valid_rate(results: List[Dict[str, Any]]) -> float:
    if not results:
        return 0.0
    return sum(1 for r in results if r.get("valid_json") is True) / len(results)

def schema_compliance_rate(results: List[Dict[str, Any]]) -> float:
    if not results:
        return 0.0
    ok = 0
    for r in results:
        try:
            CareSummary(**r.get("structured", {}))
            ok += 1
        except Exception:
            pass
    return ok / len(results)

def fact_recall_rate(results: List[Dict[str, Any]], examples_by_id: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    recalls: List[float] = []
    omitted: List[str] = []
    for r in results:
        ex = examples_by_id.get(r.get("example_id", ""))
        facts = ex.get("expected_facts", []) if ex else []
        score = _fact_recall_example(facts, _output_text(r))
        if score is None:
            continue
        recalls.append(score)
        if score < 1.0 and facts:
            omitted.append(r.get("example_id", ""))
    return {"metric": "fact_recall_heuristic", "mean_recall": round(sum(recalls) / len(recalls), 4) if recalls else 0.0,
            "examples_evaluated": len(recalls), "omitted_fact_examples": omitted[:20]}

def unsupported_claim_rate(results: List[Dict[str, Any]], examples_by_id: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    if not results:
        return {"metric": "unsupported_claim_rate_heuristic", "rate": 0.0, "examples_evaluated": 0}
    total_claims = 0
    unsupported = 0
    flagged_ids: List[str] = []
    for r in results:
        ex = examples_by_id.get(r.get("example_id", ""))
        note = ex.get("input_note", "") if ex else r.get("input_note", "")
        structured = r.get("structured", {}) or {}
        claims: List[str] = [structured.get("summary", "")]
        for obs in structured.get("observations", []) or []:
            claims.append(obs.get("statement", ""))
        for u in structured.get("unsupported_claims", []) or []:
            claims.append(u.get("statement", ""))
        for c in claims:
            if not c.strip():
                continue
            total_claims += 1
            ev = ""
            for obs in structured.get("observations", []) or []:
                if obs.get("statement") == c:
                    ev = obs.get("evidence_quote", "")
                    break
            supported = bool(ev) and quote_found_in_note(ev, note)
            if not supported:
                unsupported += 1
                flagged_ids.append(r.get("example_id", ""))
    rate = unsupported / total_claims if total_claims else 0.0
    return {"metric": "unsupported_claim_rate_heuristic", "rate": round(rate, 4), "unsupported_claims": unsupported,
            "total_claims": total_claims, "examples_evaluated": len(results), "flagged_example_ids": flagged_ids[:20]}

def uncertainty_preservation_rate(results: List[Dict[str, Any]], examples_by_id: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    total = 0
    preserved = 0
    for r in results:
        ex = examples_by_id.get(r.get("example_id", ""))
        refs = ex.get("uncertainties", []) if ex else []
        if not refs:
            continue
        total += 1
        structured_text = _output_text(r)
        structured = r.get("structured", {}) or {}
        flags = bool(structured.get("unclear_or_conflicting")) or any(o.get("uncertain") for o in structured.get("observations", []) or [])
        if flags or has_uncertainty_marker(structured_text):
            preserved += 1
    return {"metric": "uncertainty_preservation_rate", "rate": round(preserved / total, 4) if total else 0.0,
            "examples_with_uncertainty": total, "preserved": preserved}

def contradiction_preservation_rate(results: List[Dict[str, Any]], examples_by_id: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    total = 0
    preserved = 0
    for r in results:
        ex = examples_by_id.get(r.get("example_id", ""))
        contr = ex.get("known_contradictions", []) if ex else []
        if not contr:
            continue
        total += 1
        structured_text = _output_text(r)
        has_unclear = bool((r.get("structured") or {}).get("unclear_or_conflicting"))
        if has_unclear or has_uncertainty_marker(structured_text):
            preserved += 1
    return {"metric": "contradiction_preservation_rate", "rate": round(preserved / total, 4) if total else 0.0,
            "examples_with_contradictions": total, "preserved": preserved}

def latency_stats(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    lats = [r.get("latency_s", 0.0) for r in results if r.get("latency_s") is not None]
    if not lats:
        return {"metric": "latency", "count": 0}
    return {"metric": "latency", "count": len(lats), "mean_s": round(sum(lats) / len(lats), 4),
            "max_s": round(max(lats), 4), "min_s": round(min(lats), 4)}

def failure_rate(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not results:
        return {"metric": "failure_rate", "rate": 0.0, "count": 0}
    fails = sum(1 for r in results if not r.get("ok", True) or r.get("error"))
    return {"metric": "failure_rate", "rate": round(fails / len(results), 4), "failures": fails, "count": len(results)}

def evaluate_all(results: List[Dict[str, Any]], examples_by_id: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    return {"json_valid_rate": round(json_valid_rate(results), 4),
            "schema_compliance_rate": round(schema_compliance_rate(results), 4),
            "fact_recall": fact_recall_rate(results, examples_by_id),
            "unsupported_claims": unsupported_claim_rate(results, examples_by_id),
            "uncertainty_preservation": uncertainty_preservation_rate(results, examples_by_id),
            "contradiction_preservation": contradiction_preservation_rate(results, examples_by_id),
            "latency": latency_stats(results), "failure_rate": failure_rate(results)}
print("Evaluation metrics defined.")

Evaluation metrics defined.


In [10]:

def generate_report(results: List[Dict[str, Any]], metrics: Dict[str, Any], dataset_version: str,
                    model_ids: List[str], config_snapshot: Dict[str, Any]) -> str:
    lines = []
    lines.append("# Caregiver AI Lab — Experiment Report")
    lines.append("")
    lines.append(f"- Dataset version: {dataset_version}")
    lines.append("- Synthetic data notice: examples are fictional and for engineering practice only.")
    lines.append(f"- Models evaluated: {', '.join(model_ids)}")
    lines.append(f"- Examples evaluated: {len(results)}")
    lines.append("")
    lines.append("## Metrics")
    lines.append(f"- Valid JSON rate: {metrics.get('json_valid_rate')}")
    lines.append(f"- Schema compliance rate: {metrics.get('schema_compliance_rate')}")
    fr = metrics.get("failure_rate", {})
    lines.append(f"- Failure rate: {fr.get('rate')} ({fr.get('failures')}/{fr.get('count')})")
    lat = metrics.get("latency", {})
    lines.append(f"- Mean latency: {lat.get('mean_s')} s ({lat.get('count')} examples)")
    fr_claims = metrics.get("unsupported_claims", {})
    lines.append(f"- Unsupported-claim rate (heuristic): {fr_claims.get('rate')}")
    fr_fact = metrics.get("fact_recall", {})
    lines.append(f"- Fact recall (heuristic): {fr_fact.get('mean_recall')}")
    unc = metrics.get("uncertainty_preservation", {})
    lines.append(f"- Uncertainty preservation: {unc.get('rate')}")
    con = metrics.get("contradiction_preservation", {})
    lines.append(f"- Contradiction preservation: {con.get('rate')}")
    lines.append("")
    lines.append("## Limitations")
    lines.append("- Results are computed on a small synthetic dataset; they do not measure real-world clinical performance.")
    lines.append("- Fact recall and unsupported-claim rates are heuristics, not definitive correctness measures.")
    lines.append("- Do not use this tool for diagnosis or treatment decisions.")
    lines.append("")
    lines.append("## Reproducibility")
    lines.append(f"- Config: {config_snapshot}")
    return "\n".join(lines)

def save_report(markdown: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(markdown, encoding="utf-8")

def export_results_jsonl(results: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for row in results:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")

def export_review_csv(results: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = ["example_id","model_id","input_note","summary","valid_json","latency_s","error","human_label","human_notes"]
    with path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for row in results:
            structured = row.get("structured", {}) or {}
            writer.writerow({"example_id": row.get("example_id",""), "model_id": row.get("model_id",""),
                "input_note": row.get("input_note",""), "summary": structured.get("summary", row.get("summary","")),
                "valid_json": row.get("valid_json",""), "latency_s": row.get("latency_s",""),
                "error": row.get("error",""), "human_label": row.get("human_label",""), "human_notes": row.get("human_notes","")})
print("Report + export defined.")

Report + export defined.


In [11]:

write_dataset(generate_dataset(150), DATASET_PATH)
examples = load_dataset(DATASET_PATH)
val_report = validate_dataset(examples)
print(f"Examples: {val_report['count']} | Valid: {val_report['valid']} | Unique IDs: {val_report['unique_ids']}")
if val_report["errors"]:
    print("Errors:", val_report["errors"][:3])
examples_by_id = {ex["example_id"]: ex for ex in examples}
print("Sample:", examples[0]["example_id"], "|", examples[0]["language_type"], "|", examples[0]["input_note"][:80])

Examples: 150 | Valid: True | Unique IDs: 150
Sample: EG-SYN-0001 | mix | الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.


In [12]:

baseline = RuleBasedBaseline()
print("Baseline available:", baseline.available())
sample_note = examples[0]["input_note"]
res = baseline.predict(sample_note)
print("Input:", sample_note)
print("Summary:", res.summary)
import json as _json
print(_json.dumps(res.structured, ensure_ascii=False, indent=2)[:800])
print("Latency:", res.latency_s, "s | OK:", res.ok)

Baseline available:

 True
Input: الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.
Summary: الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.
{
  "summary": "الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.",
  "observations": [
    {
      "category": "medication",
      "statement": "الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.",
      "evidence_quote": "الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.",
      "uncertain": false
    }
  ],
  "unclear_or_conflicting": [],
  "unsupported_claims": [],
  "model_id": "rule-based-extractive-baseline",
  "latency_s": null,
  "valid_json": true,
  "parse_error": null
}
Latency: 0.003 s | OK: True


In [13]:

hf = HfGenerativeModel()
print("HF model:", hf.name(), "| Available:", hf.available())
if not hf.available():
    print("Load error:", hf._load_error)
else:
    res_h = hf.predict(sample_note)
    print("HF Summary:", res_h.summary)
    print("Valid JSON:", res_h.valid_json, "| Error:", res_h.error)
    print(_json.dumps(res_h.structured, ensure_ascii=False, indent=2)[:800])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


HF model: Omartificial-Intelligence-Project/AraT5v2-base-102M | Available: False
Load error: Omartificial-Intelligence-Project/AraT5v2-base-102M is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


In [14]:

limit = 30
models_to_run = [baseline] + ([hf] if hf.available() else [])
results = []
for model in models_to_run:
    for ex in examples[:limit]:
        r = model.predict(ex["input_note"])
        rec = r.to_record()
        rec["example_id"] = ex["example_id"]
        rec["language_type"] = ex["language_type"]
        results.append(rec)
metrics = evaluate_all(results, examples_by_id)
print(f"Evaluated {len(results)} runs across {[m.name() for m in models_to_run]}")
print(f"Valid JSON rate:       {metrics['json_valid_rate']}")
print(f"Schema compliance:     {metrics['schema_compliance_rate']}")
print(f"Failure rate:          {metrics['failure_rate']['rate']} ({metrics['failure_rate']['failures']}/{metrics['failure_rate']['count']})")
print(f"Mean latency:          {metrics['latency']['mean_s']} s")
print(f"Fact recall (heuristic): {metrics['fact_recall']['mean_recall']}")
print(f"Uncertainty preservation: {metrics['uncertainty_preservation']['rate']}")
print(f"Contradiction preservation: {metrics['contradiction_preservation']['rate']}")
print(f"Unsupported-claim rate (heuristic): {metrics['unsupported_claims']['rate']}")

Evaluated 30 runs across ['rule-based-extractive-baseline']
Valid JSON rate:       1.0
Schema compliance:     1.0
Failure rate:          0.0 (0/30)
Mean latency:          0.0013 s
Fact recall (heuristic): 0.3111
Uncertainty preservation: 1.0
Contradiction preservation: 0.5
Unsupported-claim rate (heuristic): 0.0


In [15]:

print("Fact recall detail:")
print("  omitted-fact examples:", metrics["fact_recall"]["omitted_fact_examples"][:10])
print()
print("Unsupported claims detail:")
uc = metrics["unsupported_claims"]
print("  unsupported/total claims:", uc["unsupported_claims"], "/", uc["total_claims"])
print("  flagged examples:", uc["flagged_example_ids"][:10])
print()
print("Uncertainty preservation:", metrics["uncertainty_preservation"])
print("Contradiction preservation:", metrics["contradiction_preservation"])
print("Latency stats:", metrics["latency"])

Fact recall detail:
  omitted-fact examples: ['EG-SYN-0001', 'EG-SYN-0002', 'EG-SYN-0003', 'EG-SYN-0004', 'EG-SYN-0005', 'EG-SYN-0006', 'EG-SYN-0007', 'EG-SYN-0008', 'EG-SYN-0009', 'EG-SYN-0010']

Unsupported claims detail:
  unsupported/total claims: 0 / 60
  flagged examples: []

Uncertainty preservation: {'metric': 'uncertainty_preservation_rate', 'rate': 1.0, 'examples_with_uncertainty': 2, 'preserved': 2}
Contradiction preservation: {'metric': 'contradiction_preservation_rate', 'rate': 0.5, 'examples_with_contradictions': 4, 'preserved': 2}
Latency stats: {'metric': 'latency', 'count': 30, 'mean_s': 0.0013, 'max_s': 0.0022, 'min_s': 0.0007}


In [16]:

worst = None
for ex in examples[:limit]:
    r = baseline.predict(ex["input_note"])
    facts = ex["expected_facts"]
    if not facts:
        continue
    norm_out = (r.summary + " " + " ".join(o.get("statement","") for o in r.structured.get("observations", []))).lower()
    hits = sum(1 for f in facts if all(w in norm_out for w in f.split() if len(w) > 2)) / len(facts)
    if worst is None or hits < worst[0]:
        worst = (hits, ex, r)

hits, worst_ex, worst_res = worst
print("Worst recall example:", worst_ex["example_id"], "| recall:", round(hits, 2))
print()
print("INPUT NOTE:")
print(" ", worst_ex["input_note"])
print()
print("REFERENCE:")
print(_json.dumps({k: worst_ex[k] for k in ["expected_facts","important_details","uncertainties","known_contradictions","expected_summary","expected_evidence_quotes"]}, ensure_ascii=False, indent=2))
print()
print("BASELINE OUTPUT:")
print(worst_res.summary)
print(_json.dumps(worst_res.structured, ensure_ascii=False, indent=2))

Worst recall example: EG-SYN-0006 | recall: 0.0

INPUT NOTE:
  ملاحظة رعاية يومية خيالية لـ أم خالد. تشكو من ألم في الركبة غير متأكدين إذا شربت الماء أم لا الطقس معتدل اليوم

REFERENCE:
{
  "expected_facts": [
    "knee pain",
    "uncertain water intake"
  ],
  "important_details": [
    "knee pain",
    "uncertain water intake"
  ],
  "uncertainties": [
    "غير متأكدين إذا شربت الماء أم لا"
  ],
  "known_contradictions": [],
  "expected_summary": "أم خالد: تشكو من ألم في الركبة، غير متأكدين إذا شربت الماء أم لا.",
  "expected_evidence_quotes": [
    "تشكو من ألم في الركبة",
    "غير متأكدين إذا شربت الماء أم لا",
    "الطقس معتدل اليوم"
  ]
}

BASELINE OUTPUT:
تشكو من ألم في الركبة غير متأكدين إذا شربت الماء أم لا الطقس معتدل اليوم
{
  "summary": "تشكو من ألم في الركبة غير متأكدين إذا شربت الماء أم لا الطقس معتدل اليوم",
  "observations": [
    {
      "category": "hydration",
      "statement": "تشكو من ألم في الركبة غير متأكدين إذا شربت الماء أم لا الطقس معتدل اليوم",
      "evide

In [17]:

print("=" * 70)
for ex in examples[:3]:
    print("ID:", ex["example_id"], "| Lang:", ex["language_type"])
    print("NOTE:", ex["input_note"])
    rb = baseline.predict(ex["input_note"])
    print("BASELINE:", rb.summary)
    if hf.available():
        rh = hf.predict(ex["input_note"])
        print("HF:", rh.summary, "| valid JSON:", rh.valid_json)
    print()

ID: EG-SYN-0001 | Lang: mix
NOTE: الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.
BASELINE: الدواء موجود في الحقيبة تشكو من ألم في الركبة م ح.

ID: EG-SYN-0002 | Lang: en
NOTE: Fictional daily care note for Um Khalid. drank a small amount of water unsure if she took the medication or not slept well all night the weather is mild today
BASELINE: drank a small amount of water unsure if she took the medication or not slept well all night the weather is mild today

ID: EG-SYN-0003 | Lang: ar
NOTE: ملاحظة رعاية يومية خيالية لـ أم خالد. ألم خفيف في البطن يأتي ويذهب أخدت الدواء في الموعد استخدمت الحمام بشكل طبيعي غير متأكدين إذا شربت الماء أم لا ش غ الطقس معتدل اليوم
BASELINE: ألم خفيف في البطن يأتي ويذهب أخدت الدواء في الموعد استخدمت الحمام بشكل طبيعي غير متأكدين إذا شربت الماء أم لا ش غ الطقس معتدل اليوم



In [18]:

export_results_jsonl(results, RESULTS_PATH)
export_review_csv(results, REVIEW_PATH)
md_report = generate_report(results, metrics, SYNTHETIC_DATASET_VERSION,
                            [m.name() for m in models_to_run],
                            {"max_new_tokens": MAX_NEW_TOKENS, "max_input_chars": MAX_INPUT_CHARS})
save_report(md_report, REPORT_PATH)
print("Written:")
for f in sorted(RESULTS_DIR.iterdir()):
    print(" ", f.name, f.stat().st_size, "bytes")
print()
print(md_report)

Written:
  experiment_report.md 864 bytes
  experiment_results.jsonl 41322 bytes
  human_review_export.csv 10264 bytes

# Caregiver AI Lab — Experiment Report

- Dataset version: v1.0-synthetic
- Synthetic data notice: examples are fictional and for engineering practice only.
- Models evaluated: rule-based-extractive-baseline
- Examples evaluated: 30

## Metrics
- Valid JSON rate: 1.0
- Schema compliance rate: 1.0
- Failure rate: 0.0 (0/30)
- Mean latency: 0.0013 s (30 examples)
- Unsupported-claim rate (heuristic): 0.0
- Fact recall (heuristic): 0.3111
- Uncertainty preservation: 1.0
- Contradiction preservation: 0.5

## Limitations
- Results are computed on a small synthetic dataset; they do not measure real-world clinical performance.
- Fact recall and unsupported-claim rates are heuristics, not definitive correctness measures.
- Do not use this tool for diagnosis or treatment decisions.

## Reproducibility
- Config: {'max_new_tokens': 256, 'max_input_chars': 2000}


In [19]:

import gradio as gr

MODEL_REGISTRY = {baseline.name(): baseline, hf.name(): hf} if hf.available() else {baseline.name(): baseline}

def run_models(note: str, model_names: List[str]) -> List[Dict[str, Any]]:
    outs = []
    for mname in model_names:
        model = MODEL_REGISTRY.get(mname)
        if model is None:
            continue
        res = model.predict(note)
        rec = res.to_record()
        rec["example_id"] = f"run-{abs(hash(note)) % 100000:05d}"
        rec["language_type"] = "mixed"
        outs.append(rec)
    return outs

def try_note(note: str, language: str, model_name: str):
    if not note or not note.strip():
        return ("Please enter a fictional care note.", "", "", "", "", "", "")
    model = MODEL_REGISTRY.get(model_name, baseline)
    result = model.predict(note)
    structured = result.structured or {}
    obs_lines = []
    for o in structured.get("observations", []):
        obs_lines.append(f"- [{o.get('category')}] {o.get('statement')} (evidence: {o.get('evidence_quote')})")
    obs_text = "\n".join(obs_lines) if obs_lines else "No observations extracted."
    unclear = structured.get("unclear_or_conflicting", [])
    unsup = structured.get("unsupported_claims", [])
    warn_parts = []
    if any(o.get("uncertain") for o in structured.get("observations", [])):
        warn_parts.append("⚠ Output marks some observations as uncertain — treat them as unconfirmed.")
    if unsup:
        warn_parts.append(f"⚠ {len(unsup)} unsupported claim(s) detected in output.")
    if not result.get("valid_json"):
        warn_parts.append("⚠ Model returned invalid JSON; showing best-effort parse.")
    warnings_html = "<br>".join(warn_parts) if warn_parts else "No warnings."
    latency = f"{result.latency_s:.3f} s" if result.latency_s is not None else "n/a"
    return (result.summary or "", obs_text,
            _json.dumps(structured, ensure_ascii=False, indent=2),
            "\n".join(o.get("evidence_quote","") for o in structured.get("observations", [])) or "—",
            warnings_html, latency, result.error or "")

def compare_models(note: str, model_names: List[str]):
    if not note or not note.strip():
        return ("Please enter a note.",) * 8
    if len(model_names) < 2:
        return ("Select at least two models.",) * 8
    outputs = []
    for mname in model_names[:4]:
        model = MODEL_REGISTRY.get(mname)
        result = model.predict(note) if model else None
        if not result:
            outputs.append((mname, "Model unavailable", "", "n/a", "error"))
            continue
        structured = result.structured or {}
        outputs.append((mname, result.summary or "(no output)",
                        _json.dumps(structured, ensure_ascii=False, indent=2)[:1500],
                        f"{result.latency_s:.3f} s" if result.latency_s else "n/a",
                        "ok" if result.ok else f"error: {result.error}"))
    while len(outputs) < 4:
        outputs.append(("", "", "", "", ""))
    return tuple(sum((list(o) for o in outputs), []))

LEARN_MARKDOWN = """
# Learn — Concepts used in this project
### Baselines
A simple non-generative method (rule-based sentence selection). Compare complex models against it.
### Tokenization
Models read tokens, not characters. Arabic + English in one note changes tokenization.
### Prompt engineering
The generative adapter wraps your note in a fixed instruction. Wording changes outputs.
### Model inference
Running the model to produce a summary. Local vs API models differ in latency, cost, privacy.
### Precision and recall
Precision = share of extracted facts that are correct. Recall = share of reference facts kept. Heuristics here.
### Hallucinations
Statements stated as fact but not present in the input note.
### Evaluation leakage
Never evaluate on examples used to design prompts.
### Train / validation / test separation
Inference only here, but hold out a test set you evaluate once.
### Reproducibility
Results depend on model weights, generation settings, dataset version.
### Why metrics can be misleading
A model can score high recall by copying long chunks without understanding.
"""

def build_app() -> gr.Blocks:
    css = """
    .arabic-note { direction: rtl; text-align: right; font-family: 'Amiri', 'Segoe UI', Tahoma, sans-serif; }
    .warn { color: #b45309; font-weight: 600; }
    .ok-text { color: #166534; font-weight: 600; }
    """
    with gr.Blocks(title="Caregiver AI Lab", css=css, theme=gr.themes.Soft()) as demo:
        gr.Markdown("# Caregiver AI Lab\nFictional Egyptian-Arabic care-note summarization — experimentation platform. Not a medical tool.")
        model_names_state = gr.State(list(MODEL_REGISTRY.keys()))
        with gr.Tabs():
            with gr.TabItem("1 · Try a Note"):
                note_in = gr.Textbox(label="Fictional daily care note", lines=4, placeholder="例如: أم خالد أكلت كمية قليلة ...")
                lang_in = gr.Dropdown(["ar", "mix", "en"], value="mix", label="Language / script")
                model_in = gr.Dropdown(choices=list(MODEL_REGISTRY.keys()), value=baseline.name(), label="Model")
                gen_btn = gr.Button("Generate summary", variant="primary")
                out_summary = gr.Textbox(label="Summary")
                out_obs = gr.Textbox(label="Observations", lines=6)
                out_struct = gr.Code(label="Structured output (JSON)", language="json")
                out_ev = gr.Textbox(label="Evidence quotes")
                out_warn = gr.HTML(label="Warnings")
                out_lat = gr.Textbox(label="Latency")
                out_err = gr.Textbox(label="Error", interactive=False)
                gen_btn.click(try_note, inputs=[note_in, lang_in, model_in],
                              outputs=[out_summary, out_obs, out_struct, out_ev, out_warn, out_lat, out_err])
            with gr.TabItem("2 · Compare Models"):
                cmp_note = gr.Textbox(label="Note to run through all models", lines=4)
                cmp_models = gr.CheckboxGroup(choices=list(MODEL_REGISTRY.keys()), value=list(MODEL_REGISTRY.keys()), label="Models to compare")
                cmp_btn = gr.Button("Run comparison", variant="primary")
                cmp_outs = []
                for i in range(4):
                    with gr.Column():
                        name = ["Model A","Model B","Model C","Model D"][i]
                        cmp_outs += [gr.Textbox(label=f"{name} — Summary"),
                                     gr.Code(label=f"{name} — Structured (JSON)", language="json"),
                                     gr.Textbox(label=f"{name} — Latency"),
                                     gr.Textbox(label=f"{name} — Status", interactive=False)]
                cmp_btn.click(compare_models, inputs=[cmp_note, cmp_models], outputs=cmp_outs)
            with gr.TabItem("3 · Evaluation Dashboard"):
                gr.Markdown("Run the evaluation cells above, then inspect results below.")
                eval_limit = gr.Number(value=30, label="Examples to evaluate", precision=0)
                eval_btn = gr.Button("Run evaluation on dataset", variant="primary")
                eval_text = gr.Markdown("No experiment has run yet.")
                eval_chart = gr.BarPlot(x="metric", y="value", title="Experiment metrics")
                def dashboard_wrapper(limit):
                    exs = examples[: max(1, min(int(limit), len(examples)))]
                    res = []
                    for model in models_to_run:
                        for ex in exs:
                            r = model.predict(ex["input_note"])
                            rec = r.to_record()
                            rec["example_id"] = ex["example_id"]
                            rec["language_type"] = ex["language_type"]
                            res.append(rec)
                    met = evaluate_all(res, examples_by_id)
                    lines = [f"Examples evaluated: {len(res)}",
                             f"Valid JSON rate: {met['json_valid_rate']}",
                             f"Schema compliance: {met['schema_compliance_rate']}",
                             f"Failure rate: {met['failure_rate']['rate']} ({met['failure_rate']['failures']}/{met['failure_rate']['count']})",
                             f"Mean latency: {met['latency']['mean_s']} s",
                             f"Fact recall (heuristic): {met['fact_recall']['mean_recall']}",
                             f"Uncertainty preservation: {met['uncertainty_preservation']['rate']}",
                             f"Unsupported-claim rate (heuristic): {met['unsupported_claims']['rate']}"]
                    import pandas as pd
                    rows = [("Valid JSON rate", met["json_valid_rate"]), ("Schema compliance", met["schema_compliance_rate"]),
                            ("Fact recall (heuristic)", met["fact_recall"]["mean_recall"]),
                            ("Uncertainty preservation", met["uncertainty_preservation"]["rate"]),
                            ("Contradiction preservation", met["contradiction_preservation"]["rate"]),
                            ("Unsupported-claim rate (heuristic)", met["unsupported_claims"]["rate"]),
                            ("Failure rate", met["failure_rate"]["rate"])]
                    return "\n".join(lines), pd.DataFrame(rows, columns=["metric", "value"])
                eval_btn.click(dashboard_wrapper, inputs=eval_limit, outputs=[eval_text, eval_chart])
            with gr.TabItem("4 · Error Analysis"):
                gr.Markdown("Inspect individual results and add human-review labels.")
                ex_note = gr.Textbox(label="Input note", elem_classes=["arabic-note"])
                ex_ref = gr.Code(label="Reference annotation (JSON)", language="json")
                ex_out = gr.Textbox(label="Model summary")
                ex_struct = gr.Code(label="Model structured output (JSON)", language="json")
                ex_issues = gr.Textbox(label="Automatically detected issues", lines=5)
                ex_label = gr.Radio(["Correct","Partially correct","Incorrect","Contains unsupported claims","Loses uncertainty","Omits important information"], label="Human review label")
                ex_notes = gr.Textbox(label="Notes", lines=2)
                gr.Markdown("*Select a result from the dashboard or run a single note above, then paste its ID here.*")
                ex_id_in = gr.Textbox(label="example_id to inspect")
                inspect_btn = gr.Button("Inspect")
                def inspect_example(eid: str):
                    ex = examples_by_id.get(eid, {})
                    # find last baseline result for this eid
                    rec = next((r for r in reversed(results) if r.get("example_id") == eid), None)
                    if not rec:
                        return ("Not found.",) * 5
                    structured = rec.get("structured", {}) or {}
                    issues = []
                    if not rec.get("ok"):
                        issues.append(f"Model error: {rec.get('error')}")
                    if not rec.get("valid_json"):
                        issues.append("Invalid JSON returned by model")
                    obs = structured.get("observations", [])
                    for o in obs:
                        if o.get("uncertain"):
                            issues.append(f"Uncertainty preserved: '{o.get('statement')}'")
                    if structured.get("unclear_or_conflicting"):
                        issues.append("Contradiction/unclear section present")
                    if ex.get("expected_facts") and not obs:
                        issues.append("No observations extracted (possible omitted facts)")
                    return (ex.get("input_note",""), _json.dumps(ex, ensure_ascii=False, indent=2),
                            rec.get("summary",""), _json.dumps(structured, ensure_ascii=False, indent=2),
                            "\n".join(issues) if issues else "No automated issues detected.")
                inspect_btn.click(inspect_example, inputs=ex_id_in,
                                  outputs=[ex_note, ex_ref, ex_out, ex_struct, ex_issues])
            with gr.TabItem("5 · Learn"):
                gr.Markdown(LEARN_MARKDOWN)
    return demo

app = build_app()
print("Gradio app defined. Call app.launch(server_name='0.0.0.0', server_port=7860, share=True) to open UI.")
print("In Colab: forward port 7860 (port icon in left sidebar) to open in browser.")

/tmp/ipykernel_38881/1643926872.py:95: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Caregiver AI Lab", css=css, theme=gr.themes.Soft()) as demo:


Gradio app defined. Call app.launch(server_name='0.0.0.0', server_port=7860, share=True) to open UI.
In Colab: forward port 7860 (port icon in left sidebar) to open in browser.


In [20]:

# Uncomment to launch the Gradio UI inside Colab (blocking cell):
# app.launch(server_name="0.0.0.0", server_port=7860, share=True)
print("To launch the UI locally:")
print("  python app.py   # if you extract app.py from this notebook")
print("Tabs: Try a Note · Compare Models · Evaluation Dashboard · Error Analysis · Learn")

To launch the UI locally:
  python app.py   # if you extract app.py from this notebook
Tabs: Try a Note · Compare Models · Evaluation Dashboard · Error Analysis · Learn


In [21]:

PASS, FAIL = [], []
def check(name, fn):
    try:
        info = fn()
        PASS.append(name)
        print(f"[PASS] {name}" + (f" - {info}" if info else ""))
    except Exception as e:
        FAIL.append(name)
        print(f"[FAIL] {name} - {type(e).__name__}: {e}")

check("Dataset loads & validates", lambda: f"{val_report['count']} examples, valid={val_report['valid']}")
check("Baseline available", lambda: f"available={baseline.available()}")
res0 = baseline.predict(examples[0]["input_note"])
check("Baseline runs", lambda: f"ok={res0.ok} | summary={res0.summary[:40]!r}")
eval_subset = []
for ex in examples[:10]:
    r = baseline.predict(ex["input_note"]).to_record()
    r["example_id"] = ex["example_id"]; r["language_type"] = ex["language_type"]
    eval_subset.append(r)
met0 = evaluate_all(eval_subset, examples_by_id)
check("Evaluation runs", lambda: f"json_valid={met0['json_valid_rate']} | schema={met0['schema_compliance_rate']} | fact_recall={met0['fact_recall']['mean_recall']:.3f}")
print()
print(f"Self-test: {len(PASS)} passed, {len(FAIL)} failed")
assert not FAIL, "Self-test failures: " + ", ".join(FAIL)
print("[ok] Caregiver AI Lab notebook checks passed.")

[PASS] Dataset loads & validates - 150 examples, valid=True
[PASS] Baseline available - available=True
[PASS] Baseline runs - ok=True | summary='الدواء موجود في الحقيبة تشكو من ألم في ا'
[PASS] Evaluation runs - json_valid=1.0 | schema=1.0 | fact_recall=0.383

Self-test: 4 passed, 0 failed
[ok] Caregiver AI Lab notebook checks passed.


## Done

All components are inline in this notebook. Run cells top-to-bottom:
1. Dataset → 2. Baseline → 3. Generative model (if available) → 4. Evaluation → 5. Error analysis → 6. Save results → 7. (optional) Gradio UI on port 7860.

Results are written to `/content/caregiver-ai-lab/results/`.

⚠️ Synthetic data only — not a medical tool.